# 07. ウォーターマーク - 遅れて届いたデータをどう扱うか

`06` までは、届いたデータをそのまま書き込んでいました。
ここからは **時間で区切って集計する** 話です。「10分ごとの売上」のような集計を考えます。

厄介なのは、データが **イベントの起きた順に届くとは限らない** ことです。
10:05に起きた注文が、10:30になってから届くことがあります。
ネットワークの遅延、端末のオフライン、上流の再送、理由はいくらでもあります。

すると「10:00〜10:10の売上」をいつ確定させるのか、という問題が出ます。
永久に待てば正確ですが、いつまでも結果が出ません。
この線引きをするのが **ウォーターマーク (水位)** です。

このノートブックで確かめること:

1. ウィンドウ集計の結果が、いつテーブルに出てくるか
2. 水位より古いデータが届くとどうなるか
3. 水位が上がると、保留されていた結果が出ること
4. 閾値をどう決めるか

**前提**: `00_setup` を実行済みであること。`01` `02` を読んでいること。


## 準備


In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils
from pyspark.sql import functions as F

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [2]:
CATALOG = "tech_survey"

# 集計結果なので gold に置く。他のトピックと混ざらないよう専用のフォルダを使う
TABLE = f"{CATALOG}.gold.sales_by_window"
LANDING = f"/Volumes/{CATALOG}/ops/landing/07_watermark"
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/07_watermark"

## 1. 集計には「いつ確定するか」の判断が要る

ストリームは終わりのないデータです。
「10:00〜10:10の件数」を出すには、その区間のデータが出揃うまで途中経過を持っておく必要があります。
これを **状態 (state)** と呼びます。

問題は、いつまで持つかです。

- 待つほど、遅れて届いたデータを拾えて正確になる
- しかし結果は遅れるし、状態が溜まり続けてメモリを圧迫する

そこで「**この時刻より古いイベントは、もう来ないことにする**」という線を引きます。これが水位です。
線より新しいものは待ち、古いものは捨てる。この一本で、正確さと待ち時間の両方が決まります。

まずデータを用意します。


In [ ]:
import json

spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

for path in (LANDING, CHECKPOINT):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass


# 何度かファイルを足すので、その部分だけまとめておく
def put_events(name: str, events: list[tuple[str, int]]) -> None:
    rows = [{"event_time": t, "amount": a} for t, a in events]
    dbutils.fs.put(f"{LANDING}/{name}.json", "\n".join(json.dumps(r) for r in rows), True)


# 1回目のデータ。最大の時刻は 10:30。
put_events(
    "events_1",
    [
        ("2026-09-13T10:00:00", 100),
        ("2026-09-13T10:03:00", 200),
        ("2026-09-13T10:12:00", 300),
        ("2026-09-13T10:30:00", 400),
    ],
)

display(spark.read.json(LANDING))

,amount,event_time
0,100,2026-09-13T10:00:00
1,200,2026-09-13T10:03:00
2,300,2026-09-13T10:12:00
3,400,2026-09-13T10:30:00


## 2. ウィンドウ集計を動かす

指定するのは2つです。

```python
.withWatermark("event_time", "5 minutes")        # 水位 = これまでに見た最大の event_time - 5分
.groupBy(F.window("event_time", "10 minutes"))   # 10分ごとに区切る
```

`5 minutes` は「**最大でこれくらいの遅れなら待つ**」という許容量です。
現在時刻ではなく **届いたデータの中の最大時刻** から逆算される点に注意してください。
データが来なければ水位も上がりません。

そして **`append` で書き出す場合、ウィンドウの結果はそれが閉じてからしか出ません。**
閉じるのは水位がウィンドウの終わりを越えたときです。
一度書いた行は後から直せないので、確定するまで出せないからです。

いまのデータの最大時刻は 10:30 です。どのウィンドウが出てくるか予想してみてください。


In [ ]:
# 同じストリームを3回動かすので、定義をまとめておく
def run_stream():
    # ストリームの定義 (Auto Loader)
    events = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
        .load(LANDING)
        .withColumn("event_time", F.col("event_time").cast("timestamp"))  # JSONに型が無いのでキャストする
    )

    # aggregate する前に df に watermark を付ける
    windowed = (
        events.withWatermark("event_time", "5 minutes")  # 水位 = 見た最大の event_time - 5分
        .groupBy(F.window("event_time", "10 minutes"))  # 10分ごとのウィンドウに切る
        .agg(
            F.count("*").alias("event_count"),
            F.sum("amount").alias("total_amount"),
        )
    )

    # ウィンドウの集計結果を Delta Lake に書き出す
    query = (
        windowed.writeStream.option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(TABLE)  # append で書き出す。確定したウィンドウだけが追加される
    )
    query.awaitTermination()
    return query

In [5]:
query = run_stream()

# どのウィンドウが確定して出てきたかを見る
display(
    spark.table(TABLE)
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_count",
        "total_amount",
    )
    .orderBy("window_start")
)

,window_start,window_end,event_count,total_amount
0,2026-09-13 10:00:00,2026-09-13 10:10:00,2,300.0
1,2026-09-13 10:10:00,2026-09-13 10:20:00,1,300.0


10:30 のイベントは入っていないはずです。

水位は `10:30 - 5分 = 10:25` の位置にあります。
`[10:30, 10:40)` のウィンドウは終わりが 10:40 なので、まだ越えられていません。
**まだ来ていないデータがあるかもしれない** ので、結果を出さずに状態として抱えています。

すでに、送られたきたデータは以下の通り
```plain
amount	event_time
100	2026-09-13T10:00:00
200	2026-09-13T10:03:00
300	2026-09-13T10:12:00
400	2026-09-13T10:30:00  ← こいつが集計されていなかった
```

## 3. 遅れて届いたデータ

ここに2件足します。

- **10:05 のイベント** … 25分遅れて届いた。水位 (10:25) より古い
- **10:32 のイベント** … まだ開いている `[10:30, 10:40)` に入る

10:05 は `[10:00, 10:10)` に入るはずですが、そのウィンドウはもう確定して書き出されています。
どうなるか予想してください。


In [6]:
# 2回目のデータ。10:05 は大きく遅れて届いたことにする
put_events(
    "events_2",
    [
        ("2026-09-13T10:05:00", 999),
        ("2026-09-13T10:32:00", 500),
    ],
)

# 集計を実行
query = run_stream()

# 状態の様子を見る。水位より古くて捨てられた行数がここに出る
for p in query.recentProgress:
    for op in p.stateOperators:
        print(f"batchId={p.batchId}  捨てられた行={op.numRowsDroppedByWatermark}  保持中={op.numRowsTotal}")

batchId=2  捨てられた行=1  保持中=1
batchId=3  捨てられた行=0  保持中=1


In [7]:
# [10:00, 10:10) の合計が 300 のままか、999 が足されたかを確認する
display(
    spark.table(TABLE)
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_count",
        "total_amount",
    )
    .orderBy("window_start")
)

,window_start,window_end,event_count,total_amount
0,2026-09-13 10:00:00,2026-09-13 10:10:00,2,300.0
1,2026-09-13 10:10:00,2026-09-13 10:20:00,1,300.0


10:05 の行は **捨てられました**。`numRowsDroppedByWatermark` がその件数です。

理由は `append` の性質です。一度テーブルに書いた行は後から直せません。
`[10:00, 10:10)` の結果はすでに出してしまったので、いまさら足すと
**同じウィンドウの行が2つできてしまいます。**
だから水位を越えた時点で、そのウィンドウに属するデータは受け付けなくなります。

捨てられても **エラーにはなりません**。`06` の `txnVersion` と同じで、黙って進みます。
気づくには `numRowsDroppedByWatermark` を見るしかありません。監視すべき指標のひとつです。


## 4. 水位が上がると、保留が出る

`[10:30, 10:40)` はまだ出ていないはずです。水位が 10:40 を越えていないからです。

水位は「見た最大の時刻 - 5分」なので、**新しいデータが来ないと上がりません。**
11:00 のイベントを1件足して、押し上げてみます。


In [8]:
# 3回目のデータ。これで水位が 10:55 まで上がる
put_events("events_3", [("2026-09-13T11:00:00", 600)])

query = run_stream()

display(
    spark.table(TABLE)
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_count",
        "total_amount",
    )
    .orderBy("window_start")
)

,window_start,window_end,event_count,total_amount
0,2026-09-13 10:00:00,2026-09-13 10:10:00,2,300.0
1,2026-09-13 10:10:00,2026-09-13 10:20:00,1,300.0
2,2026-09-13 10:30:00,2026-09-13 10:40:00,2,900.0


保留されていた `[10:30, 10:40)` が出てきたはずです。3章で足した 10:32 も、ここに入っています。
一方 `[11:00, 11:10)` は、今度はこれが最新なので保留に回ります。

ここが分かりにくいところです。
**時計が進んだから出たのではなく、新しいデータが来たから出ました。**
データが途絶えると、最後のウィンドウは出ないまま残り続けます。


## 5. 閾値をどう決めるか

`withWatermark` の `5 minutes` をどう決めるか、という話です。トレードオフはこの3つだけです。

| | 長くする | 短くする |
|---|---|---|
| 遅延データ | 拾える | 捨てる |
| 結果が出るまで | 遅い | 速い |
| 状態のサイズ | 大きい | 小さい |

基本は **実際の遅れを測って決める** ことです。
`event_time` と処理時刻の差を記録しておいて、その分布を見ます。
99%が3分以内に届いているなら、5分あたりが妥当な線になります。

そのうえで、捨てて困るかどうかで調整します。

- **捨てて困らない** (概況を早く知りたいダッシュボードなど) → 短くする
- **捨てると困る** (請求、在庫など) → 長くする

ただし「長くすれば安全」ではありません。状態が増え、結果も遅れます。
**絶対に取りこぼせないなら、ストリーミングの集計だけで済ませようとしないほうがよい** です。
生データを残しておいて、`04` でやった `replaceWhere` で後から作り直すほうが素直になります。


## 考えてみる

- `3.` で捨てられた 10:05 の行を、どうしても集計に入れたい場合はどうしますか
- 水位はなぜ「現在時刻 - 5分」ではなく「見た最大の `event_time` - 5分」なのでしょうか
- ウィンドウを 10分 から 1日 に変えると、何が変わりますか


### 答え

**Q1. 捨てられた行を入れたい**

ストリーミングの集計だけでは無理です。確定して書き出した後だからです。手は2つあります。

- **閾値を伸ばす**。ただし全ウィンドウの確定が遅れ、状態も増える。1件のために全体を遅くすることになる
- **後からバッチで作り直す**。生データ (Bronze) を捨てずに残しておいて、
  1日1回などその範囲を `replaceWhere` で計算し直す (`04`)

実務では後者が多いです。ストリーミングでは速報値を出し、
確定値は後からバッチで上書きする、という二段構えにします。

**Q2. なぜ現在時刻ではないのか**

過去のデータを流し直したときに壊れるからです。

1年前のログをまとめて取り込む場合、現在時刻を基準にすると水位は常に「今」の近くにあり、
**流し込んだデータが全部古すぎて捨てられます。**
データの中の時刻を基準にすれば、再処理でも本番と同じ結果になります。
`06` の「番号は処理した時刻ではなく処理対象から決める」と同じ考え方です。

副作用として「データが来ないと水位が上がらない」という性質が出ますが、
再現性のほうが重要、という判断です。

**Q3. ウィンドウを1日にすると**

結果が出るのが最大1日遅れます。`[9/13, 9/14)` の結果は、水位が 9/14 を越えるまで出ません。
状態も1日分を抱え続けることになります。

ウィンドウの長さは「どれだけ待てるか」を直接決めます。
集計の粒度というより **結果の鮮度** の設定だと考えたほうがよさそうです。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# for path in (LANDING, CHECKPOINT):
#     try:
#         dbutils.fs.rm(path, True)
#     except NotFound:
#         pass